# Colab 21 — Natural-language probe: does the AA encoder transfer the *edit-distance operation* to English?

A **diagnostic, not a thesis headline.** The frozen AA encoder (synthetic-AA training,
identical recipe to colab19/20) is fed **English-derived** strings over the *same 20-letter
alphabet*, to ask one thing:

> Does the AA-trained encoder transfer the edit-distance operation to English-like strings,
> or does its learned geometry depend on AA-like character statistics?

**Protocol (mirrors the colab19 AA-synthetic diagnostic; only the *seed source* changes).**
1. Load English source text -> uppercase -> **char-filter to the 20 AA letters** (drop every
   char not in `ACDEFGHIKLMNPQRSTVWY`, including spaces / the 6 missing letters B J O U X Z).
2. Slice the filtered stream into length 50-200 windows = **English seeds**.
3. Perturb each seed (sub/ins/del, edits drawn from the AA alphabet) to a target normLev.
4. Compute **exact normLev on the filtered strings actually fed** (fully self-consistent).
5. Feed both into the **frozen AA encoder**; compare raw L2-sim to true normLev.

Run side-by-side with **random-AA seeds** (the colab19 baseline) so English-seed vs
AA-seed transfer is directly comparable. Metrics: Spearman + isotonic OOF RMSE, full-range
and high-band (>= 0.70).

**Reading (modest):** decent Spearman -> the encoder learned something operation-like beyond
protein strings. Collapse -> the geometry leans on AA-like character statistics. Either way a
probe.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy --quiet

In [ ]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
print('device:', device)

## 2. Constants (encoder recipe = colab19/20; AA encoder only)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN, MAX_LEN_SEQ, MAX_LEN = 50, 200, 200
N_TRAIN, NUM_EPOCHS, BATCH_SIZE, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
BIN_NAMES = ['far', 'mid', 'high']

N_FULL_PROBE = 5000      # full-range pairs per seed-source
N_HIGH_PROBE = 4000      # high-band-only pairs per seed-source
N_FOLDS = 5
SEED_TRAIN_AA = 42
SEED_PROBE = 2026        # pair-generation RNG (English + AA baseline share it for fairness)

# English source text. Swap this URL for any plain-text corpus you prefer.
GUTENBERG_URL = 'https://www.gutenberg.org/files/1342/1342-0.txt'  # Pride and Prejudice (public domain)

## 3. Helpers + architecture + AA training (frozen afterwards)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_aa_seed(rng):
    length = int(rng.integers(MIN_LEN, MAX_LEN_SEQ + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def bin_idx_for(x, band_low):
    return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

def make_pairs(seed_fn, n_pairs, rng, lo=0.0, hi=1.0):
    """t ~ U[lo,hi] -> perturbation pairs. seed_fn(rng)->str supplies the seed source.
    Edits always drawn from AA_ALPHABET (matches training); normLev computed on the exact
    strings fed."""
    pairs = []; attempts = 0; max_attempts = n_pairs * 8
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn(rng)
        if seed is None or len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(lo, hi)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, AA_ALPHABET, rng)
        if 1 <= len(other) <= MAX_LEN:
            lab = norm_lev(seed, other)
            if lab >= lo:
                pairs.append((seed, other, lab))
    return pairs


class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K, self.pad_idx = K, pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, 3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(K)
        self.fc = nn.Linear(hidden2 * K, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1); h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)

class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
                                  nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b): return self.head(torch.abs(self.encoder(a) - self.encoder(b)))

class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]

def train_aa_encoder(train_seed=SEED_TRAIN_AA, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed); rng = np.random.default_rng(train_seed)
    print(f'===== Training AA encoder (seed={train_seed}, synthetic-AA perturbation pairs) =====')
    pairs = make_pairs(random_aa_seed, N_TRAIN, rng)
    dl = DataLoader(PairDatasetCE(pairs, BAND_LOW_AA), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval(); return model

model_aa = train_aa_encoder()

## 4. English corpus — load, char-filter to the 20 AA letters, build the seed sampler

In [ ]:
def load_english_stream(url=GUTENBERG_URL):
    raw = urllib.request.urlopen(url, timeout=60).read().decode('utf-8', 'ignore')
    upper = raw.upper()
    filtered = ''.join(c for c in upper if c in AA_SET)   # drop spaces + non-AA letters (B J O U X Z)
    return raw, upper, filtered

raw, upper, EN_STREAM = load_english_stream()
print(f'raw chars       : {len(raw):,}')
print(f'uppercased chars: {len(upper):,}')
print(f'AA-filtered     : {len(EN_STREAM):,}  ({len(EN_STREAM)/max(1,len(upper)):.1%} of uppercased kept)')
assert len(EN_STREAM) > MAX_LEN_SEQ * 50, 'filtered English stream too short for diverse windows'

# letter-frequency sanity: English (filtered) vs uniform-AA
from collections import Counter
cnt = Counter(EN_STREAM); tot = sum(cnt.values())
top = sorted(cnt.items(), key=lambda kv: -kv[1])[:8]
print('top filtered-English letters:', [(c, f'{n/tot:.1%}') for c, n in top])

def english_seed(rng, stream=EN_STREAM):
    L = int(rng.integers(MIN_LEN, MAX_LEN_SEQ + 1))
    start = int(rng.integers(0, len(stream) - L))
    return stream[start:start + L]

## 5. Diagnostic machinery (predicted L2-sim, isotonic OOF RMSE, Spearman) — from colab19

In [ ]:
@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def isotonic_oof(true, pred, n_folds=N_FOLDS, seed=SEED):
    true = np.asarray(true, float); pred = np.asarray(pred, float); n = len(true)
    if n < n_folds * 2 or np.std(pred) == 0: return np.nan
    oof = np.full(n, np.nan)
    for tr, te in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(pred):
        ir = IsotonicRegression(out_of_bounds='clip').fit(pred[tr], true[tr])
        oof[te] = ir.predict(pred[te])
    return float(np.sqrt(np.nanmean((true - oof) ** 2)))

def spearman_of(true, pred):
    if len(true) < 4 or np.std(pred) == 0 or np.std(true) == 0: return np.nan
    return float(spearmanr(pred, true).correlation)

def fmt(x, d=3):
    return 'n/a' if (x is None or (isinstance(x, float) and np.isnan(x))) else f'{x:.{d}f}'

## 6. Generate pairs — English seeds vs random-AA seeds (shared RNG seed for fairness)

In [ ]:
rng_en  = np.random.default_rng(SEED_PROBE)
rng_aa  = np.random.default_rng(SEED_PROBE)

en_full = make_pairs(english_seed,   N_FULL_PROBE, rng_en)
en_high = make_pairs(english_seed,   N_HIGH_PROBE, rng_en, lo=BAND_HIGH)
aa_full = make_pairs(random_aa_seed, N_FULL_PROBE, rng_aa)
aa_high = make_pairs(random_aa_seed, N_HIGH_PROBE, rng_aa, lo=BAND_HIGH)

for name, pr in [('English full', en_full), ('English high', en_high),
                 ('AA full', aa_full), ('AA high', aa_high)]:
    labs = np.array([p[2] for p in pr])
    print(f'{name:<14} n={len(pr):>5}  normLev in [{labs.min():.3f}, {labs.max():.3f}]  mean={labs.mean():.3f}')

## 7. Verdict — Spearman + isotonic OOF RMSE, full-range and high-band (>= 0.70)

In [ ]:
def evaluate(pairs):
    true = np.array([p[2] for p in pairs])
    pred = pred_sim_strings(model_aa, [p[0] for p in pairs], [p[1] for p in pairs])
    return true, pred

def summarize(label, full_pairs, high_pairs):
    tf, pf = evaluate(full_pairs)
    th, ph = evaluate(high_pairs)
    row = dict(label=label,
               full_rho=spearman_of(tf, pf), full_rmse=isotonic_oof(tf, pf),
               high_rho=spearman_of(th, ph), high_rmse=isotonic_oof(th, ph),
               high_pred_mean=float(ph.mean()), high_pred_std=float(ph.std()))
    return row, (tf, pf), (th, ph)

en_row, en_full_xy, en_high_xy = summarize('English-seed', en_full, en_high)
aa_row, aa_full_xy, aa_high_xy = summarize('AA-seed (colab19 baseline)', aa_full, aa_high)

print('=' * 92)
print('COLAB21 — natural-language transfer probe (frozen AA encoder)')
print('=' * 92)
print(f'{"seed source":<28}{"full rho":>10}{"full iso-RMSE":>15}{"high rho":>10}{"high iso-RMSE":>15}')
print('-' * 92)
for r in [aa_row, en_row]:
    print(f"{r['label']:<28}{fmt(r['full_rho']):>10}{fmt(r['full_rmse']):>15}"
          f"{fmt(r['high_rho']):>10}{fmt(r['high_rmse']):>15}")
print('\nhigh-band = pairs with true normLev >= 0.70. iso-RMSE is in normLev units (after the')
print('best monotone recalibration). Compare English-seed to the AA-seed baseline row.')

In [ ]:
# Sub-band breakdown of the high band (where does English transfer bite?)
print(f'{"source":<14}{"band":<14}{"n":>7}{"pred_mean":>11}{"spearman":>10}')
for label, (th, ph) in [('AA-seed', aa_high_xy), ('English', en_high_xy)]:
    for lo, hi in [(0.70, 0.80), (0.80, 0.90), (0.90, 1.001)]:
        m = (th >= lo) & (th < hi)
        if m.sum() >= 4:
            print(f'{label:<14}[{lo:.2f},{hi:.2f})  {int(m.sum()):>7}{ph[m].mean():>11.3f}'
                  f'{fmt(spearman_of(th[m], ph[m])):>10}')

## 8. Scatter — English-seed vs AA-seed, predicted L2-sim vs true normLev

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (label, (tf, pf)) in zip(axes, [('AA-seed (baseline)', aa_full_xy),
                                        ('English-seed', en_full_xy)]):
    ax.scatter(pf, tf, s=4, alpha=0.12)
    if np.std(pf) > 0:
        ir = IsotonicRegression(out_of_bounds='clip').fit(pf, tf)
        grid = np.linspace(pf.min(), pf.max(), 200)
        ax.plot(grid, ir.predict(grid), 'g-', lw=2, label='isotonic')
    ax.plot([0, 1], [0, 1], 'r--', lw=1, label='y = x')
    ax.axhline(BAND_HIGH, color='gray', ls=':', lw=1)
    ax.set_xlabel('predicted L2-sim'); ax.set_ylabel('true normLev')
    ax.set_title(f'{label}: rho={spearman_of(tf, pf):.3f}')
    ax.legend()
plt.suptitle('Edit-distance operation transfer: English-derived vs random-AA seeds')
plt.tight_layout(); plt.savefig('colab21_nl_probe.png', dpi=120); plt.show()

## How to read this notebook

**The comparison is English-seed vs AA-seed, same frozen encoder.** The AA-seed row is a **side-by-side AA baseline computed in this notebook** (fresh encoder + fresh perturbation pairs), not a strict reproduction of colab19's exact frozen diagnostic — read it as the in-notebook control / ceiling for this run, and expect numbers in the same ballpark as colab19's synthetic-AA verdict (rho ~0.85, iso-RMSE ~0.04) rather than identical. The English-seed row is the probe.

- **English high-band rho close to the AA-seed baseline** -> the encoder transferred the
  edit-distance *operation* to English-like strings: it is reading position-pattern structure,
  not AA-specific character statistics. Supports the position-pattern-hashing mechanism.
- **English high-band rho collapses (≪ baseline)** -> the geometry leans on AA-like character
  frequencies; English's different letter statistics break it. That would bound the
  "alphabet-agnostic operation" claim.
- **iso-RMSE** says how tight the recoverable value is after a monotone recalibration; read it
  alongside rho (rho = order preserved; RMSE = magnitude recoverable).

**Scope / caveats.** Diagnostic only, not a thesis headline. Char-filtering drops spaces and the
6 non-AA letters (B J O U X Z), so "English" here = English letter-frequency/adjacency structure
over the 20-letter alphabet, not literal text. Perturbation edits are AA-random (matches
training); only the *seed* carries English structure — the clean isolation of seed-statistics
transfer. Single corpus (swap GUTENBERG_URL to test robustness). Deferred: colab22
masked-adaptive-pooling ablation.